# 📝 파이썬 기초 과제 LV3(통합) — 파일·예외·모듈·클래스

> 파이썬 기초에서 배운 **전부**를 모아 실제로 쓸 수 있는 작은 프로그램을 만듭니다.
> 함수 분리 · 메뉴 루프 · 예외 방어 · 파일 저장(영속화)을 모두 담아요.

## 구성
1. **미니 가계부** — 지출을 기록하고 JSON 으로 저장/불러오기
2. **단어장** — 단어를 모으고 퀴즈로 외우기 (CSV 저장)
3. **회원 포인트 카드** — 클래스와 JSON 저장을 엮어 만드는 작은 프로그램

> 각 문제는 **v1 → v2 → v3** 로 단계를 나눠 놓았어요. 한 번에 완성하려 하지 말고 **단계별로** 키워 가세요.
> 읽기는 `data/`, 쓰기는 `output/` (없으면 `os.makedirs` 로 생성).

## 1. 미니 가계부 프로그램
**배경**: 매일 쓴 돈을 기록하고, 분류별로 얼마 썼는지 확인하고, 껐다 켜도 기록이 남는 나만의 가계부.

### 단계별 요구사항
**v1 — 함수로 나누기**
- `add_expense(ledger, amount, category)` : 지출 하나(`{"분류":…, "금액":…}`)를 리스트에 추가
- `category_totals(ledger)` : 분류별 합계 딕셔너리 반환 (예: `{"식비": 8000, "교통": 1400}`)

**v2 — 저장/불러오기 (영속화)**
- `save_ledger(path, ledger)` : `output/가계부.json` 으로 저장 (`json.dump`, `ensure_ascii=False`)
- `load_ledger(path)` : 파일이 있으면 불러오고, **없으면 빈 리스트** 반환 (`try/except FileNotFoundError`)

**v3 — 메뉴 루프**
- `run(path)` : 아래 메뉴를 반복. 시작할 때 기존 기록을 불러오고, 모든 숫자 입력은 예외 방어.
```
1. 지출 추가   2. 전체 목록   3. 분류별 합계   4. 저장   0. 종료
```

**v4 — 패키지로 분리**
- 저장/불러오기와 집계 기능을 `ledger/` 패키지(폴더)로 나눕니다.
  - `ledger/storage.py` : `save_ledger`, `load_ledger`
  - `ledger/report.py` : `category_totals`
  - `ledger/__init__.py` : 세 함수를 재노출하고 `__all__` 로 공개 목록을 선언
- 그 뒤 `import ledger` 로 불러와 `ledger.save_ledger(...)`, `ledger.category_totals(...)` 처럼 쓸 수 있어야 합니다.

**예시 흐름**
```
메뉴 선택: 1
금액: 8000
분류(1식비 2교통 3문화 4기타): 1
추가됨: 식비 8000원
메뉴 선택: 3
식비: 8000원
메뉴 선택: 0
종료합니다.
```

<details><summary>힌트</summary>

```text
접근방법:
- 데이터만 다루는 순수 함수(추가·집계), 저장/불러오기, 사용자와 대화하는 메뉴 루프로 나눈다.
- 마지막으로 저장·집계 기능을 ledger 패키지(폴더)로 분리해 정리한다.

세부구현:
1. add_expense — 분류와 금액을 담은 딕셔너리 하나를 ledger 리스트에 추가
2. category_totals — ledger 를 순회하며 분류별로 금액을 누적한 딕셔너리를 반환 (처음 보는 분류는 0에서 시작)
3. save_ledger — 저장 폴더를 준비한 뒤 ledger 를 JSON으로 기록 (한글이 깨지지 않게)
4. load_ledger — 파일을 열어 JSON을 읽어 반환하되, 파일이 없으면 FileNotFoundError 를 잡아 빈 리스트 반환
5. run — 시작할 때 load_ledger 로 기존 기록을 불러오고, 메뉴 번호를 입력받아 반복
   5-1. 번호 입력은 정수 변환을 try/except 로 방어
   5-2. 0이면 종료, 1은 지출 추가(분류 번호 1~4를 식비/교통/문화/기타로 변환), 2는 전체 목록, 3은 분류별 합계, 4는 save_ledger 로 저장
6. 패키지로 분리 — 저장/불러오기와 집계 함수를 ledger 폴더로 옮긴다
   6-1. ledger/storage.py 에 save_ledger, load_ledger 를 둔다
   6-2. ledger/report.py 에 category_totals 를 둔다
   6-3. ledger/__init__.py 에서 세 함수를 재노출하고 __all__ 에 이름을 적는다
   6-4. 현재 폴더를 import 경로에 추가한 뒤 import ledger 로 불러와 쓴다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]  (대화형 run() 은 아래 셀에서 직접 실행 — 여기선 입력이 필요 없는 부분만 검증)
# 1) v1 순수 함수
ledger = []
add_expense(ledger, 5000, "식비")
add_expense(ledger, 3000, "교통")
add_expense(ledger, 2000, "식비")
assert len(ledger) == 3
totals = category_totals(ledger)
assert totals["식비"] == 7000 and totals["교통"] == 3000

# 2) v2 저장·불러오기 (파일이 실제로 만들어지고 다시 읽히는지)
save_ledger("output/가계부_test.json", ledger)
reloaded = load_ledger("output/가계부_test.json")
assert len(reloaded) == 3
assert load_ledger("output/없는파일.json") == []   # 없는 파일은 빈 목록
print("✅ 문제1 통과!")

In [ ]:
# 직접 실행해 보려면 아래 주석을 풀고 실행하세요. 메뉴에서 0을 눌러 종료합니다.
# (지출을 추가한 뒤 4번으로 저장하면 output/가계부.json 에 기록이 남아요.)
# run("output/가계부.json")

In [ ]:
# [자가채점]  v4 패키지 분리 확인 (import ledger 후 동작 + __all__ 점검)
import sys, os
sys.path.insert(0, os.getcwd())
import ledger

records = [{"분류": "식비", "금액": 5000}, {"분류": "교통", "금액": 3000}, {"분류": "식비", "금액": 2000}]
ledger.save_ledger("output/가계부_pkg.json", records)
loaded = ledger.load_ledger("output/가계부_pkg.json")
assert len(loaded) == 3, "저장·불러오기가 되어야 해요"

totals = ledger.category_totals(records)
assert totals["식비"] == 7000 and totals["교통"] == 3000

assert set(ledger.__all__) >= {"save_ledger", "load_ledger", "category_totals"}, "__all__ 에 세 함수를 공개하세요"
print("✅ v4 패키지 분리 통과!")

## 2. 단어장 프로그램
**배경**: 새 단어를 모아 두고, 퀴즈로 반복해서 외우고, 파일로 저장해 두는 나만의 단어장.

### 단계별 요구사항
**v1 — 단어 관리**
- `load_words(path)` : CSV(`영어,뜻`)를 읽어 딕셔너리 리스트로 (없으면 빈 리스트)
- `add_word(words, english, meaning)` : 단어 하나 추가
- `save_words(path, words)` : `output/단어장.csv` 로 저장

**v2 — 퀴즈 모드**
- `quiz(words, num_questions=3)` : 무작위로 문제를 내고, 사용자의 답이 뜻과 같으면 점수 +1. 마지막에 `score/num_questions` 출력.

**v3 — 메뉴 루프**
- `run(path)` : `1.단어 추가  2.목록  3.퀴즈  4.저장  0.종료` 반복.
- `4.저장` 을 고르면 지금까지의 단어를 `output/단어장.csv` 로 저장합니다. (추가한 단어가 사라지지 않게!)

**예시 흐름**
```
메뉴: 3
'apple' 의 뜻은? 사과
정답!
점수: 1/3
메뉴: 4
저장 완료 → output/단어장.csv
메뉴: 0
종료합니다.
```

<details><summary>힌트</summary>

```text
접근방법:
- 단어 관리(불러오기·추가·저장) 함수, 퀴즈 함수, 메뉴 루프로 나눈다.

세부구현:
1. load_words — CSV를 DictReader 로 읽어 반환하되 파일이 없으면 빈 리스트 반환 (단어 하나는 '영어'와 '뜻' 두 키를 가진 딕셔너리)
2. add_word — '영어'와 '뜻' 키를 가진 딕셔너리를 words 리스트에 추가
3. save_words — 저장 폴더를 준비하고 헤더(영어,뜻)를 먼저 쓴 뒤 단어들을 기록
4. quiz — 무작위로 num_questions 개 단어를 뽑아 뜻을 물어보고, 입력한 답이 실제 뜻과 같으면 점수를 올려 마지막에 점수를 출력
5. run — 메뉴 번호를 입력받아 반복. 0이면 종료, 1은 단어 추가, 2는 목록, 3은 퀴즈, 4는 save_words 로 output/단어장.csv 에 저장
```

</details>

In [ ]:
# [제공 코드] 단어장 실습용 CSV 를 준비합니다. (이미 있으면 같은 내용으로 덮어써요)
import os
os.makedirs("data", exist_ok=True)
word_pairs = [
    ("apple", "사과"), ("banana", "바나나"), ("computer", "컴퓨터"),
    ("python", "파이썬"), ("teacher", "선생님"), ("student", "학생"),
    ("water", "물"), ("book", "책"), ("music", "음악"), ("future", "미래"),
]
with open("data/단어_샘플.csv", "w", encoding="utf-8", newline="") as f:
    f.write("영어,뜻\n")
    for en, meaning in word_pairs:
        f.write(f"{en},{meaning}\n")
print("단어_샘플.csv 준비 완료 (총", len(word_pairs), "개)")

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]  (대화형 run() 은 아래 셀에서 직접 실행 — 여기선 입력이 필요 없는 부분만 검증)
words = load_words("data/단어_샘플.csv")
assert len(words) >= 5, "단어가 5개 이상 있어야 해요"
assert "영어" in words[0] and "뜻" in words[0]

add_word(words, "hello", "안녕")
assert words[-1]["영어"] == "hello"

save_words("output/단어장.csv", words)
reloaded = load_words("output/단어장.csv")
assert any(w["영어"] == "hello" for w in reloaded), "저장·불러오기가 되어야 해요"
print("✅ 문제2 통과!")

In [ ]:
# 직접 실행해 보려면 아래 주석을 풀고 실행하세요. 메뉴에서 0을 눌러 종료합니다.
# (3번 퀴즈에서 뜻을 직접 입력해 보고, 4번으로 저장하면 output/단어장.csv 에 남아요.)
# run("data/단어_샘플.csv")

## 3. 회원 포인트 카드 (클래스 + JSON 저장)
**배경**: 오늘 배운 **클래스**와 **파일 저장(JSON)** 을 한데 엮어 작은 프로그램을 만듭니다. 포인트를 적립·사용하고, 그 상태를 파일에 저장해요.

**만들 것**
1. `PointCard` 클래스
   - `__init__(self, name)` : `self.name` 저장, `self.points` 를 `0` 으로 시작
   - `earn(self, amount)` : 포인트를 `amount` 만큼 적립
   - `use(self, amount)` : 포인트가 충분하면 `amount` 만큼 차감하고 `True` 반환, 부족하면 아무것도 안 하고 `False` 반환
   - `save(self, path)` : `{"name": 이름, "points": 포인트}` 를 JSON 파일로 저장 (`ensure_ascii=False`, 폴더 없으면 makedirs)
2. 메뉴 루프: `1.적립  2.사용  3.저장  0.종료` — **입력은 모두 숫자**. 적립·사용 금액도 숫자로 입력받아 위 메서드를 호출

**예시**
```
card = PointCard("민준")
card.earn(5000)
card.use(2000)   ->  True   (남은 포인트 3000)
card.use(999999) ->  False  (부족, 그대로 3000)
```

<details><summary>힌트</summary>

```text
접근방법:
- 상태(name, points)를 클래스에 담고, earn/use/save 메서드로 다룬다. 메뉴 루프가 사용자 입력을 받아 메서드를 부른다.

세부구현:
1. class PointCard: __init__ 에서 self.name = name, self.points = 0
2. earn 은 self.points += amount
3. use 는 amount 가 self.points 보다 크면 False, 아니면 self.points -= amount 후 True
4. save 는 폴더가 포함된 경로(예: output/카드.json)로 저장 — os.makedirs 후 json.dump({"name": self.name, "points": self.points}, ...) (ensure_ascii=False)
5. 메뉴 루프: while True 로 번호를 받아 0 종료, 1 적립, 2 사용, 3 저장 을 분기
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]  (메뉴 루프 없이 메서드 동작만 검증합니다)
import os, json
card = PointCard("민준")
card.earn(5000)
assert card.points == 5000, "earn 으로 적립돼야 해요"
assert card.use(2000) is True and card.points == 3000, "충분하면 차감하고 True"
assert card.use(999999) is False and card.points == 3000, "부족하면 그대로 두고 False"
card.save("output/포인트카드_test.json")
with open("output/포인트카드_test.json", encoding="utf-8") as f:
    data = json.load(f)
assert data["name"] == "민준" and data["points"] == 3000
print("✅ 문제3 통과!")

**메뉴 루프 (직접 실행해 보세요)**

In [ ]:
# 여기에 코드를 작성하세요